# Test a mesh-diffusion checkpoint

The sibling of `test-mesh-normal.ipynb` for `config_set: mesh_diffusion`. It
loads the **same** splits a training run used -- same merged config, same seed,
same `random_split` -- runs the full reverse trajectory from each LOD1
condition, and shows input / ground truth / generated side by side for `train`,
`val` and `test`.

The train split is a memorization check, not a score: those buildings are the
ones the weights were fit on, so the gap between train and val/test is the
overfitting readout, and a train building that still comes out wrong is a
capacity or optimization problem rather than a generalization one. On this
branch that check earns its place -- a face-set diffusion model can collapse to
"mark every slot absent", which scores well on `gen_coord_mse` and produces no
mesh at all, and the failure looks identical on every split.

Sampling is `n_steps` forward passes per batch (50 by default, 100 with
classifier-free guidance on), so keep `N_SHOW` small -- it is per split.

In [ ]:
import sys
import tempfile
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import torch
from omegaconf import OmegaConf

import plotly.graph_objects as go
import plotly.io as pio

from src.dataset.mesh_set_dataset import mesh_set_collate_fn
from src.eval.mesh_metrics import mesh_metrics
# `_scaffold_hook` and `fixed_seed` are what MeshSetEvalCallback runs; importing
# them rather than reimplementing is what keeps this notebook's numbers the same
# quantity the run logged.
from src.eval.mesh_set_eval import _scaffold_hook, fixed_seed
from src.models.mesh_diffusion_module import MeshDiffusionModule
from src.models.mesh_set_postprocess import faces_to_mesh
from src.train_mesh_diffusion import MeshSetDataModule
from src.utils.initialization import load_config
from src.visualize_mesh import cityjson_figure_from_mesh, mesh_figure, side_by_side

pio.renderers.default = "notebook"

## The experiment to test

Arm configs carry only the axes they set, so `run_experiments.sh --diff-stage N`
merges each one over `configs/mesh-diff-base.yaml` before launching. `BASE` does
the same merge here, in the same order -- point it at `None` for a config that
is already self-contained (`mesh-diff-smoke.yaml`).

`CKPT = None` picks the newest checkpoint under the run directory the merged
config names. Note that checkpoints are *selected* on `val_gen_chamfer_m`, which
is scored from the EMA weights, so `WEIGHTS = "ema"` is what the selection
believed it was choosing; `"live"` is the same run's no-EMA control.

In [ ]:
ARM = ROOT / "configs" / "mesh-diff-a1-unet-mse.yaml"
BASE = ROOT / "configs" / "mesh-diff-base.yaml"   # None if ARM is self-contained
CKPT = None            # None = newest .ckpt under the run directory
SPLITS = ("train", "val", "test")   # which splits to generate from
N_SHOW = 3             # buildings PER SPLIT; each batch is n_steps forward passes
N_STEPS = None         # None = mesh_diffusion.eval_steps, the tier-3 budget
WEIGHTS = "ema"        # "ema" (what selected the checkpoint) or "live"
SEED = 1234            # MeshSetEvalCallback's default, so numbers line up

# The merge run_experiments.sh does, written out and read back through
# `load_config` so the schema defaults and type coercion are byte-for-byte what a
# real run got -- not a second implementation of the same merge.
if BASE is not None:
    merged = OmegaConf.merge(OmegaConf.load(BASE), OmegaConf.load(ARM))
    CONFIG = Path(tempfile.mkdtemp()) / f"{ARM.stem}.merged.yaml"
    OmegaConf.save(merged, CONFIG)
else:
    CONFIG = ARM

cfg = load_config(CONFIG, [])

# Dataset paths in a config are relative to the REPO ROOT, because that is where
# a training run is launched from. This notebook runs from notebooks/, and
# `MeshSetDataModule` takes the whole cfg and resolves the path itself -- so it
# has to be made absolute here rather than passed in at the call site the way
# the autoregressive notebook passes `ROOT / ...` to `MeshDataModule`.
cfg.mesh_data.dataset_dir = str(ROOT / cfg.mesh_data.dataset_dir)

d = cfg.mesh_diffusion
num_bins = cfg.mesh_data.num_bins
run_dir = ROOT / cfg.logging.save_dir / cfg.logging.experiment_name / cfg.logging.run_name

if CKPT is None:
    ckpts = sorted(run_dir.rglob("*.ckpt"), key=lambda p: p.stat().st_mtime)
    if not ckpts:
        raise FileNotFoundError(f"No .ckpt under {run_dir} -- set CKPT explicitly.")
    CKPT = ckpts[-1]

print(f"arm {ARM.name}, run {cfg.logging.run_name}")
print(f"data  {cfg.mesh_data.dataset_dir}")
print(f"order={d.order} denoiser={d.denoiser} process={d.process} "
      f"target={d.target} loss={d.loss} state={d.state}")
print(f"scaffold={d.scaffold.enabled} guidance={d.guidance} "
      f"slot_budget={d.slot_budget} num_bins={num_bins}")
print(f"checkpoint {Path(CKPT).relative_to(ROOT)}")

## The same splits

`MeshSetDataModule` is imported from `src/train_mesh_diffusion.py` rather than
rebuilt, so the dataset arguments, the split fractions and the `cfg.seed`
generator are the ones the run used and position *i* of a split here is the same
building the run saw at position *i*.

The split is seeded the same way `MeshDataModule` seeds its own, which is what
lets a diffusion arm and an autoregressive arm on the same `dataset_dir` be
compared building by building.

In [ ]:
datamodule = MeshSetDataModule(cfg)
datamodule.setup()

splits = {name: getattr(datamodule, f"{name}_dataset") for name in SPLITS}

print(f"train={len(datamodule.train_dataset)} val={len(datamodule.val_dataset)} "
      f"test={len(datamodule.test_dataset)}")
print("generating from: " + ", ".join(f"{n} ({min(N_SHOW, len(s))})"
                                      for n, s in splits.items()))

In [ ]:
# `cfg=` is required. The module saves the RESOLVED CONFIG DICT as its
# hyperparameters while `__init__` takes the `Config` object, so Lightning's
# stored hparams cannot rebuild it on their own -- and `validate_combination`
# inside `__init__` re-runs the same gate a training run passed, which is what
# catches a checkpoint being read under an arm it was not trained as.
model = MeshDiffusionModule.load_from_checkpoint(CKPT, cfg=cfg, map_location="cpu")
model.eval()
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

steps = N_STEPS or d.eval_steps
weights = WEIGHTS if (WEIGHTS == "live" or model.ema_active) else "live"
if weights != WEIGHTS:
    print("EMA holds nothing in this checkpoint -- falling back to live weights")
print(f"{sum(p.numel() for p in model.denoiser.parameters()) / 1e6:.1f}M denoiser "
      f"parameters, {steps} reverse steps, {weights} weights on {device}")

## Generate

One batch per split, at the pinned `slot_budget` -- the same `width` the eval
collate uses, so a sample is a function of the weights alone and not of which
other buildings landed in the batch. The scaffold hook is the callback's own, so
an arm with `scaffold.enabled` is sampled the way it was scored.

`faces_to_mesh` turns the `[10, F]` sample into a mesh: slots whose presence
channel is not positive are dropped, coordinates are snapped to the bin grid,
degenerate faces are dropped, and the rest is welded. Its `stats` are kept --
a mesh with a plausible chamfer and 400 vertices where 60 belong is a different
failure from a wrong shape.

The `ceil` mesh is the *ground truth* through the same `faces_to_mesh`. That is
the branch's representation floor: no sample can score better than it, so a
chamfer is only readable against it.

In [ ]:
def to_metres(mesh, center, scale):
    """(verts, faces) in the unit box -> metres, undoing the LOD1-box frame."""
    verts, faces = mesh
    return verts * scale + center, faces


def raw_pair(dataset, i):
    """The ((verts, faces), (verts, faces)) LOD1/LOD2 pair in metres behind
    position `i` of a split, reached through the `Subset` the split is."""
    base, idx = dataset, i
    while hasattr(base, "dataset"):
        idx, base = base.indices[idx], base.dataset
    return base.mesh_pair(idx)


results = []

with fixed_seed(SEED), model.using_weights(weights):
    for split, dataset in splits.items():
        picks = list(range(min(N_SHOW, len(dataset))))
        if not picks:
            continue
        items = [dataset[i] for i in picks]
        batch = mesh_set_collate_fn(items, multiple_of=8, width=d.slot_budget,
                                    num_bins=num_bins)
        batch = {k: v.to(device) if torch.is_tensor(v) else v
                 for k, v in batch.items()}

        with torch.no_grad():
            sampled = model.generate(batch, n_steps=steps,
                                     scaffold=_scaffold_hook(batch, cfg))

        for k, i in enumerate(picks):
            center = batch["center"][k].cpu().numpy()
            scale = batch["scale"][k].cpu().numpy()
            lod1, gt = raw_pair(dataset, i)

            gen_v, gen_f, stats = faces_to_mesh(sampled[k], num_bins,
                                                snap=d.snap_before_weld,
                                                min_area=d.min_face_area)
            ceil_v, ceil_f, _ = faces_to_mesh(batch["x"][k], num_bins,
                                              snap=d.snap_before_weld,
                                              min_area=d.min_face_area)
            results.append({
                "split": split, "name": batch["ids"][k],
                "lod1": lod1, "gt": gt,
                "ceil": to_metres((ceil_v, ceil_f), center, scale),
                "gen": to_metres((gen_v, gen_f), center, scale),
                "stats": stats,
            })
            print(f"{split}/{batch['ids'][k]}: lod1 {len(lod1[1])} tris | "
                  f"gt {len(gt[1])} | ceiling {len(ceil_f)} | gen {len(gen_f)}"
                  + ("  <- EMPTY SAMPLE" if not len(gen_f) else ""))

## Input / ground truth / generated

Top row: the raw triangle meshes in metres. LOD1 and LOD2 are the corpus meshes
themselves -- unlike the autoregressive notebook there is no tokenizer round trip
in the middle, because this branch never tokenizes: `faces_to_mesh`'s snap is the
only quantization, and it is applied to the *sample*, not to the target.

Bottom row: the same geometry through `mesh_to_cityjson` -- coplanar triangles
merged back into polygons and coloured by semantic surface (blue ground, orange
roof, grey wall). An empty bottom panel means the writer refused the mesh, which
on this branch is the common case long before the geometry is good.

The printed metrics are `run_mesh_set_eval`'s, per building rather than averaged:
the unprefixed row is the sample against the true LOD2, and `ceiling` is the true
LOD2 through `faces_to_mesh` against itself -- the `gt_` row of the eval, and the
floor the sample's chamfer has to be read against.

In [ ]:
figs = {}


def city_panel(mesh):
    """`cityjson_figure_from_mesh`, with a blank panel where it cannot run.

    A free-running sample can come back empty or as a soup the writer rejects,
    and letting that raise would cost every figure after it.
    """
    if not len(mesh[1]):
        return go.Figure(), 0
    try:
        return cityjson_figure_from_mesh(*mesh)
    except Exception as exc:
        print(f"  CityJSON write-back failed: {exc}")
        return go.Figure(), 0


_metric = dict(taus=list(d.taus), n_points=d.n_points, voxel_m=d.voxel_m, seed=SEED)

for r in results:
    split, name, lod1, gt, gen = r["split"], r["name"], r["lod1"], r["gt"], r["gen"]
    if len(gen[1]):
        row = mesh_metrics(gen, gt, **_metric)
        print(split, name, {k: round(v, 4) for k, v in sorted(row.items())
                            if np.isfinite(v)})
    else:
        # Not a missing measurement: an all-absent sample is a real failure mode
        # of the presence channel, and dropping it would flatter the average.
        print(split, name, "EMPTY SAMPLE -- no surface, every metric undefined")
    if len(r["ceil"][1]):
        ceil = mesh_metrics(r["ceil"], gt, **_metric)
        print("   ceiling", {k: round(v, 4) for k, v in sorted(ceil.items())
                             if np.isfinite(v)})

    meshes = [(lod1, "#898781"), (gt, "#2a78d6"), (gen, "#eda100")]
    top = [mesh_figure(*mesh, color=color) for mesh, color in meshes]
    bottom = [city_panel(mesh) for mesh, _ in meshes]

    figs[f"{split}/{name}"] = side_by_side(
        [top, [fig for fig, _ in bottom]],
        [f"[{split}] LOD1 input, {len(lod1[1])} tris",
         f"[{split}] LOD2 ground truth, {len(gt[1])} tris",
         f"[{split}] LOD2 generated, {len(gen[1])} tris"]
        + [f"CityJSON, {n} surfaces" if n else "CityJSON: none" for _, n in bottom],
    )

print(f"\n{len(figs)} figures built -- display them one at a time in the last cell.")

## What the sample lost on its way to a mesh

Where the autoregressive notebook asks whether MeshAnything's post-processing
would recover anything, this branch has no such question: `faces_to_mesh`
already welds, drops duplicate faces and fixes winding on every sample. What is
worth reading instead is *where the slots went*, which is the same `stats` dict
`run_mesh_set_eval` logs and the only thing that separates several very
different failures that all show up as a bad chamfer:

| column | what a large number means |
|---|---|
| `absent` | slots marked no-object. The whole face-count mechanism -- `slot_budget` minus this is what the model decided to build. |
| `degenerate` | all three corners landed in one bin. Coordinates are collapsing, not just wrong. |
| `duplicate` | the same triangle generated more than once. Slots are not differentiating. |
| `faces` / `verts` | what survived. `verts` far above `3 x faces / 2` means corners are not meeting, so the surface is a soup rather than a shell. |

`slots` is the pinned `slot_budget`, so the four drop counts plus `faces` account
for it exactly, up to the faces the weld itself removed.

In [ ]:
import pandas as pd

rows = []
for r in results:
    s = r["stats"]
    rows.append({
        "split": r["split"], "building": r["name"],
        "slots": s["n_slots"],
        "absent": s["n_dropped_absent"],
        "degenerate": s["n_dropped_degenerate"],
        "duplicate": s["n_dropped_duplicate"],
        "faces": s["n_faces"], "verts": s["n_verts"],
        "gt_faces": len(r["gt"][1]),
        "ceil_faces": len(r["ceil"][1]),
    })

table = pd.DataFrame(rows).set_index(["split", "building"])

n_empty = int((table["faces"] == 0).sum())
print(f"{n_empty}/{len(table)} samples produced no mesh at all")
print("mean faces: generated "
      f"{table['faces'].mean():.1f} vs ground truth {table['gt_faces'].mean():.1f}")
# The ceiling is the target through the same pipeline, so a gap here is the
# representation's, not the model's -- and it should be small.
lost = table["gt_faces"] - table["ceil_faces"]
print(f"ceiling loses {lost.mean():.1f} faces on average to the snap "
      "(representation cost, not model error)")
print(f"   N_SHOW={N_SHOW} per split is a spot check, not a rate")

table

In [ ]:
# One at a time: each figure carries six 3-D scenes, and displaying all of them
# is what makes this notebook multi-megabyte on disk.
print(list(figs))

KEY = list(figs)[0]
figs[KEY]